# Method chapter visualizations

Clean, self-made figures for the **stochastic action representations** in the
PPO section (Method, Section *Representing the Deterministic TFv6 Outputs as
Stochastic Distributions*). Three figures:

1. **Speed bins** -- the categorical target-speed distribution and its
   probability-weighted average (the executed target speed).
2. **Ego-pivot route rotation** -- the single scalar route correction that
   rotates the whole trajectory about the ego centre.
3. **Exploration contrast** -- independent per-waypoint noise (jagged) vs. the
   ego-pivot rotation (smooth), motivating why exploration is constrained to
   the single mode.

Constants match the codebase: speed classes from
`lead/training/config_training.py`; route geometry (`target_first_distance=2.5`,
`spacing=1.0`) from `rl_finetuning/tfv6_rl/action_noise_codec.build_route_basis`.
Figures save to `figures/` as both PDF and PNG; copy the PNGs into the Overleaf
`Images/` folder.


### Style
Identical Computer Modern + Paul Tol *bright* setup as the other notebooks.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt

# --- Fonts/colours: identical to result_graphs.ipynb and
# residual_correction_distribution.ipynb (Computer Modern via matplotlib's
# bundled "cmr10", Paul Tol "bright" palette) so these figures match the thesis.
USE_TEX = False
SERIF = ["cmr10", "CMU Serif", "Latin Modern Roman", "STIXGeneral", "DejaVu Serif"]
PALETTE = {"blue": "#4477AA", "red": "#EE6677", "green": "#228833", "yellow": "#CCBB44",
           "cyan": "#66CCEE", "purple": "#AA3377", "grey": "#BBBBBB"}
CYCLE = [PALETTE[k] for k in ("blue", "red", "green", "purple", "cyan", "yellow")]

BASE = 11
mpl.rcParams.update({
    "text.usetex": USE_TEX,
    "font.family": "serif", "font.serif": SERIF,
    "mathtext.fontset": "cm", "axes.unicode_minus": False,
    "axes.formatter.use_mathtext": True,
    "font.size": BASE, "axes.titlesize": BASE + 1, "axes.labelsize": BASE,
    "xtick.labelsize": BASE - 1, "ytick.labelsize": BASE - 1, "legend.fontsize": BASE - 1,
    "axes.linewidth": 0.8, "axes.edgecolor": "#444444",
    "axes.spines.top": False, "axes.spines.right": False, "axes.axisbelow": True,
    "axes.grid": True, "grid.color": "#CCCCCC", "grid.linewidth": 0.6, "grid.alpha": 0.7,
    "xtick.direction": "out", "ytick.direction": "out",
    "xtick.major.size": 3, "ytick.major.size": 3,
    "lines.linewidth": 1.8, "legend.frameon": False, "legend.handlelength": 1.6,
    "figure.figsize": (5.9, 3.6), "figure.dpi": 120,
    "savefig.dpi": 300, "savefig.bbox": "tight",
    "figure.constrained_layout.use": True,
    "axes.prop_cycle": mpl.cycler(color=CYCLE),
})

FIG_DIR = Path("figures")
def save_fig(fig, name, formats=("pdf", "png")):
    FIG_DIR.mkdir(parents=True, exist_ok=True)
    for fmt in formats:
        fig.savefig(FIG_DIR / f"{name}.{fmt}")
    print("saved:", ", ".join(f"{name}.{f}" for f in formats), "->", FIG_DIR.resolve())

print("matplotlib", mpl.__version__, "| serif ->",
      Path(mpl.font_manager.findfont("serif")).name)

## Figure 1 -- Categorical target speed
The eight fixed speed bins carry a softmax probability; the controller executes
the probability-weighted average $\mathbb{E}[v]=\sum_i p_i v_i$ (red dashed).

In [ ]:
# TFv6 target-speed classes (m/s), from lead/training/config_training.py
SPEED_BINS = np.array([0.0, 4.0, 8.0, 10.0, 13.88888888, 16.0, 17.77777777, 20.0])

# Illustrative predicted distribution: most mass on 4 and 8 m/s, some on 0,
# little elsewhere -- a cautious "creep / moderate speed" prediction.
probs = np.array([0.15, 0.34, 0.30, 0.10, 0.05, 0.03, 0.02, 0.01])
probs = probs / probs.sum()
Ev = float((probs * SPEED_BINS).sum())   # probability-weighted average -> executed target speed

# Plot on a categorical axis so the (uneven) bins are equal-width and touch;
# the real m/s value of each bin is the tick label.
idx = np.arange(len(SPEED_BINS))
fig, ax = plt.subplots(figsize=(6.0, 3.3))
ax.bar(idx, probs, width=1.0, color=PALETTE["blue"], alpha=0.9,
       edgecolor="white", lw=1.2, zorder=3)

# Executed target speed: dashed marker at the interpolated categorical position.
ev_x = float(np.interp(Ev, SPEED_BINS, idx))
ax.axvline(ev_x, color=PALETTE["red"], lw=2.6, ls="--", zorder=4)
ax.annotate(r"executed target speed $\approx %.1f$ m/s" % Ev,
            (ev_x, probs.max() * 1.14), xytext=(8, 0), textcoords="offset points",
            color=PALETTE["red"], fontsize=12.5, ha="left", va="bottom", zorder=6)

ax.set_xticks(idx)
ax.set_xticklabels([("%.0f" % b) if abs(b - round(b)) < 1e-2 else ("%.1f" % b) for b in SPEED_BINS])
ax.set_xlabel(r"Target-speed bin $v_i$ (m/s)")
ax.set_ylabel("Probability")
ax.set_xlim(-0.5, len(idx) - 0.5)
ax.set_ylim(0, probs.max() * 1.30)
ax.grid(axis="x", visible=False)
save_fig(fig, "method_ppo_speed_bins")
plt.show()

## Figure 2 -- Ego-pivot route rotation
A single coefficient $c\in[-1,1]$ rotates the whole route about the ego centre:
each point is deflected laterally in proportion to its longitudinal distance, so
the pivot sits at the ego and even the nearest (low-speed) control point turns.

In [ ]:
# Route geometry (ego frame), from action_noise_codec.build_route_basis:
#   route point i is at longitudinal distance x_i = target_first_distance + i*spacing
#   (target_first_distance = 2.5 m, spacing = 1.0 m).
from matplotlib.patches import FancyBboxPatch

N = 10
x = 2.5 + np.arange(N) * 1.0          # longitudinal distance ahead (m)
y_base = np.zeros(N)                  # straight base route, for a clean illustration
pivot = x / x.max()                   # 0 at the ego, 1 at the farthest point
MAX_LAT = 2.5                         # lateral travel (m) of the farthest point at c = +/-1

coeff_colors = {-1.0: PALETTE["green"], 0.0: PALETTE["blue"], 1.0: PALETTE["red"]}

fig, ax = plt.subplots(figsize=(5.4, 5.7))

# --- ego vehicle footprint (top view), centred at the pivot (ego centre) ---
CAR_W, CAR_L = 1.9, 4.5
ax.add_patch(FancyBboxPatch((-CAR_W / 2, -CAR_L / 2), CAR_W, CAR_L,
                            boxstyle="round,pad=0,rounding_size=0.45",
                            linewidth=1.4, edgecolor="#555555",
                            facecolor="#DDDDDD", zorder=2))
ax.text(0, -CAR_L / 2 + 0.35, "ego\nvehicle", ha="center", va="bottom",
        fontsize=8.5, color="#666666")

# --- routes: base and the two extreme corrections ---
for c, col in coeff_colors.items():
    y = y_base + c * MAX_LAT * pivot
    if c == 0:
        ax.plot(y, x, "-o", color=col, lw=2.6, ms=4.2, zorder=5)
    else:
        ax.plot(y, x, "-o", color=col, lw=1.9, ms=3.2, zorder=4)

# pivot marker
ax.plot(0, 0, marker="x", color="black", ms=8, mew=1.7, zorder=6)
ax.annotate("pivot (ego centre)", (0, 0), textcoords="offset points", xytext=(10, 0),
            fontsize=8.5, va="center")

# --- control points on the base route: the single point the controller reads,
#     nearer at low speed and further ahead at high speed ---
i_lo, i_hi = 0, 7
ax.scatter([0, 0], [x[i_lo], x[i_hi]], s=85, facecolors="none",
           edgecolors=PALETTE["purple"], lw=1.8, zorder=7)
ax.annotate("control point\n(low speed)", xy=(0, x[i_lo]), xytext=(2.05, 1.1),
            textcoords="data", color=PALETTE["purple"], fontsize=8.5, ha="left", va="center",
            arrowprops=dict(arrowstyle="->", color=PALETTE["purple"], lw=1.0, shrinkB=7), zorder=8)
ax.annotate("control point\n(high speed)", xy=(0, x[i_hi]), xytext=(2.95, x[i_hi]),
            textcoords="data", color=PALETTE["purple"], fontsize=8.5, ha="left", va="center",
            arrowprops=dict(arrowstyle="->", color=PALETTE["purple"], lw=1.0, shrinkB=7), zorder=8)

# end labels
ax.annotate("base route\n$c=0$", (0, x.max()), color=PALETTE["blue"], fontsize=9,
            ha="center", va="bottom", xytext=(0, 6), textcoords="offset points")
ax.annotate("max left\ncorrection\n$c=-1$", (-MAX_LAT, x.max()), color=PALETTE["green"],
            fontsize=9, ha="right", va="bottom", xytext=(-2, 4), textcoords="offset points")
ax.annotate("max right\ncorrection\n$c=+1$", (MAX_LAT, x.max()), color=PALETTE["red"],
            fontsize=9, ha="left", va="bottom", xytext=(2, 4), textcoords="offset points")

ax.set_xlabel("Lateral offset (m)")
ax.set_ylabel("Longitudinal distance ahead (m)")
ax.set_xlim(-MAX_LAT * 1.55, MAX_LAT * 2.1)
ax.set_ylim(-CAR_L / 2 - 0.7, x.max() + 2.2)
ax.set_aspect("equal")
save_fig(fig, "method_ppo_route_rotation")
plt.show()

## Figure 3 -- Why a single rotation, not free noise
Independent per-waypoint perturbations (left) give jagged, unrealizable paths;
the single ego-pivot rotation (right) keeps every sampled trajectory smooth and
physically meaningful.

In [ ]:
# Why exploration is constrained to the single rotation rather than free noise.
from matplotlib.patches import FancyBboxPatch

rng = np.random.default_rng(1)
N = 10
x = 2.5 + np.arange(N) * 1.0
y_base = np.zeros(N)
pivot = x / x.max()
K = 12  # number of sampled trajectories per panel

CAR_W, CAR_L = 1.9, 4.5
def draw_ego(ax):
    ax.add_patch(FancyBboxPatch((-CAR_W / 2, -CAR_L / 2), CAR_W, CAR_L,
                                boxstyle="round,pad=0,rounding_size=0.45",
                                linewidth=1.2, edgecolor="#555555",
                                facecolor="#DDDDDD", zorder=1))

fig, axes = plt.subplots(1, 2, figsize=(6.6, 4.8), sharey=True)

# (left) independent per-waypoint Gaussian noise -> jagged, unrealizable paths
sigma_ind = 0.6
for _ in range(K):
    y = y_base + rng.normal(0.0, sigma_ind, size=N)
    axes[0].plot(y, x, "-", color=PALETTE["grey"], alpha=0.7, lw=1.0, zorder=3)
axes[0].plot(y_base, x, "-o", color=PALETTE["blue"], lw=2.4, ms=4, zorder=5)
axes[0].set_title("Independent per-waypoint noise")

# (right) single ego-pivot rotation coefficient -> smooth, plausible trajectories
for _ in range(K):
    c = float(np.clip(rng.normal(0.0, 0.45), -1.0, 1.0))
    y = y_base + c * 2.5 * pivot
    axes[1].plot(y, x, "-", color=PALETTE["grey"], alpha=0.7, lw=1.0, zorder=3)
axes[1].plot(y_base, x, "-o", color=PALETTE["blue"], lw=2.4, ms=4, zorder=5,
             label="Base route")
axes[1].set_title("Ego-pivot rotation (single scalar)")

for ax in axes:
    draw_ego(ax)
    ax.set_xlabel("Lateral offset (m)")
    ax.set_xlim(-3, 3)
    ax.set_ylim(-CAR_L / 2 - 0.6, x.max() + 1.0)
    ax.set_aspect("equal")
axes[0].set_ylabel("Longitudinal distance ahead (m)")
save_fig(fig, "method_ppo_route_exploration")
plt.show()